# Bài thực hành Phân tích Dữ liệu và Học sâu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
np.random.seed(42)


## Câu 1

In [ ]:
n=500
df_khachhang=pd.DataFrame({
'MaKH':[f'KH{i:03d}' for i in range(1,n+1)],
'Tuoi':np.random.randint(18,71,n).astype(float),
'ThuNhap':np.random.uniform(5e6,5e7,n),
'GioiTinh':np.random.choice(['Nam','Nữ'],n),
'ThanhPho':np.random.choice(['Hà Nội','Đà Nẵng','TP.HCM'],n)
})
df_khachhang['TongChiTieu']=df_khachhang['ThuNhap']*np.random.uniform(0.3,0.8,n)+np.random.normal(0,2e6,n)
df_khachhang.loc[np.random.choice(n,10,False),'Tuoi']=np.nan
df_khachhang.loc[np.random.choice(n,15,False),'GioiTinh']=np.nan
idx=np.random.choice(n,5,False)
df_khachhang.loc[idx,'ThuNhap']=np.random.uniform(1.5e8,2e8,5)
df_khachhang.head()

## Câu 2

In [ ]:
print(df_khachhang.isna().sum())
df_khachhang['Tuoi']=df_khachhang['Tuoi'].fillna(df_khachhang['Tuoi'].median())
df_khachhang['GioiTinh']=df_khachhang['GioiTinh'].fillna(df_khachhang['GioiTinh'].mode()[0])

## Câu 3

In [ ]:
df_khachhang=pd.concat([df_khachhang,pd.get_dummies(df_khachhang['ThanhPho'],prefix='TP')],axis=1)
df_khachhang.head()

## Câu 4

In [ ]:
Q1=df_khachhang['ThuNhap'].quantile(.25);Q3=df_khachhang['ThuNhap'].quantile(.75)
IQR=Q3-Q1
low=Q1-1.5*IQR;up=Q3+1.5*IQR
df_khachhang=df_khachhang[(df_khachhang['ThuNhap']>=low)&(df_khachhang['ThuNhap']<=up)]
print(df_khachhang.shape)

## Câu 5

In [ ]:
sc=MinMaxScaler()
df_khachhang['TongChiTieu_Scaled']=sc.fit_transform(df_khachhang[['TongChiTieu']])

## Câu 6

In [ ]:
df_phu=df_khachhang[(df_khachhang.GioiTinh=='Nữ')&(df_khachhang.Tuoi>30)&(df_khachhang.ThanhPho=='Hà Nội')]
df_phu.head()

## Câu 7

In [ ]:
print(df_khachhang.groupby('ThanhPho')['TongChiTieu'].agg(['mean','sum']))

## Câu 8

In [ ]:
bins=[18,30,45,60,100]
labels=['18-30','31-45','46-60','Trên 60']
df_khachhang['NhomTuoi']=pd.cut(df_khachhang['Tuoi'],bins=bins,labels=labels,include_lowest=True)
df_khachhang[['Tuoi','NhomTuoi']].head()

## Câu 9

In [ ]:
corr=df_khachhang[['Tuoi','ThuNhap','TongChiTieu']].corr()
sns.heatmap(corr,annot=True,cmap='coolwarm')
plt.show()

## Câu 10

In [ ]:
sns.scatterplot(data=df_khachhang,x='ThuNhap',y='TongChiTieu',hue='GioiTinh')
plt.show()

## Câu 11

In [ ]:
(x_train,y_train),(x_test,y_test)=fashion_mnist.load_data()
x_train=x_train.astype('float32')/255
x_test=x_test.astype('float32')/255
x_train=x_train.reshape((-1,28,28,1))
x_test=x_test.reshape((-1,28,28,1))
model=Sequential([
Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)),
MaxPooling2D((2,2)),
Flatten(),
Dense(64,activation='relu'),
Dense(10,activation='softmax')
])
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.fit(x_train,y_train,epochs=5,validation_split=0.1)
loss,acc=model.evaluate(x_test,y_test)
print('Test Accuracy:',acc)